# Ejercicio 11: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.
## Michael Perugachi

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import nltk
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
from bs4 import BeautifulSoup
import requests

In [3]:
# Descargar stopwords si no están disponibles
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# URL principal
main_url = "https://www.allrecipes.com/recipes-a-z-6735880"

In [9]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,/;q=0.8",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}

## Parte 2: Obtener los datos deseados
### Buscar dentro del contenido HTML y extraer la información.

In [10]:
# Listas para almacenar los datos
data = []

# Hacer la solicitud inicial
response = requests.get(main_url, headers=headers)

In [11]:
# Verificar si la solicitud fue exitosa
if response.status_code == 200:
    # Parsear el HTML de la página principal
    soup = BeautifulSoup(response.text, 'html.parser')

    # Buscar los enlaces principales
    main_items = soup.find_all('li', class_='mntl-link-list__item')

    # Usar tqdm para mostrar la barra de progreso
    for main_item in tqdm(main_items, desc="Procesando enlaces principales"):
        main_link = main_item.find('a')
        if main_link:
            main_text = main_link.get_text(strip=True)  # Texto del enlace principal
            main_href = main_link['href']  # URL del enlace principal

            # Hacer una solicitud a la URL del enlace principal
            inner_response = requests.get(main_href, headers=headers)

            # Verificar si la solicitud fue exitosa
            if inner_response.status_code == 200:
                # Parsear el HTML de la página interna
                inner_soup = BeautifulSoup(inner_response.text, 'html.parser')

                # Buscar los enlaces internos (según la clase indicada en tu imagen)
                inner_items = inner_soup.find_all('a', class_='comp mntl-card-list-items mntl-universal-card mntl-document-card mntl-card card card--no-image')

                for inner_item in inner_items:
                    inner_text = inner_item.get_text(strip=True) if inner_item else None
                    inner_href = inner_item['href'] if 'href' in inner_item.attrs else None

                    # Agregar datos al conjunto de resultados
                    data.append({
                        'Texto': main_text,
                        'URL': main_href,
                        'Receta': inner_text,
                        'Receta_URL': inner_href
                    })

else:
    print(f"Error al acceder a la página principal: {response.status_code}")

# Crear un DataFrame con los datos recopilados
df = pd.DataFrame(data)

Procesando enlaces principales: 100%|██████████| 378/378 [03:44<00:00,  1.68it/s]


In [12]:
df

,Texto,URL,Receta,Receta_URL
0,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Jalapeno Popper Pinwheels6Ratings,https://www.allrecipes.com/air-fryer-jalapeno-...
1,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Miso-Glazed Salmon,https://www.allrecipes.com/air-fryer-miso-glaz...
2,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Hoisin Salmon,https://www.allrecipes.com/air-fryer-hoisin-sa...
3,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Stuffed Peppers1Rating,https://www.allrecipes.com/air-fryer-stuffed-p...
4,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Croissant Breakfast Boats2Ratings,https://www.allrecipes.com/air-fryer-croissant...
...,...,...,...,...
19019,Zucchini Breads,https://www.allrecipes.com/recipes/348/bread/q...,Healthier Mom's Zucchini Bread64Ratings,https://www.allrecipes.com/recipe/222078/healt...
19020,Zucchini Breads,https://www.allrecipes.com/recipes/348/bread/q...,"Zucchini Bread, Pumpkin Style13Ratings",https://www.allrecipes.com/recipe/152103/zucch...
19021,Zucchini Breads,https://www.allrecipes.com/recipes/348/bread/q...,Gluten-Free Zucchini Bread (or Muffins)11Ratings,https://www.allrecipes.com/recipe/244775/glute...
19022,Zucchini Breads,https://www.allrecipes.com/recipes/348/bread/q...,Cherry-Zucchini Bread2Ratings,https://www.allrecipes.com/recipe/277978/cherr...


In [13]:
# Reduciremos los datos a 100
df = df[:100]

In [14]:
# Agregar nuevas columnas vacías al DataFrame
df['Nombre'] = ''
df['Ingredientes'] = ''
df['Image'] = ''
df['Preparación'] = ''
df['Descripción'] = ''
df['Información_nutricional'] = ''
df['Detalles'] = ''
df['Rating'] = ''
df['Comentarios'] = ''

/tmp/ipython-input-2796870401.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Nombre'] = ''
/tmp/ipython-input-2796870401.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ingredientes'] = ''
/tmp/ipython-input-2796870401.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-v

In [15]:
!pip install beautifulsoup4 selenium

## Parte 3: Obtener enlaces relacionados
### Encontrar links a otras recetas para completar el corpus

In [16]:
# Iterar por cada URL
from IPython.display import Image, display
for i, url in enumerate(tqdm(df['Receta_URL'], desc="Procesando URLs")):
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        ### Nombre (Título de la receta principal)
        h1 = soup.find('h1')
        df.at[i, 'Nombre'] = h1.get_text(strip=True) if h1 else 'No encontrado'

        ### Descripción (og:title)

        title = soup.find("meta", {"property": "og:title"})["content"]
        description = soup.find("meta", attrs={"name": "description"})["content"]
        df.at[i, 'Descripción'] = description

        rating = soup.select_one("#mm-recipes-review-bar__rating_1-0")
        df.at[i, 'Rating'] = rating.get_text(strip=True) if rating else 'No encontrado'


        ### Ingredientes
        ingredientes_list = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
        ingredientes_text = [ingrediente.get_text(strip=True) for ingrediente in ingredientes_list]
        df.at[i, 'Ingredientes'] = ', '.join(ingredientes_text) if ingredientes_text else 'No encontrado'

        ### Preparación / Pasos
        pasos_section = soup.find_all("li", class_="mntl-sc-block-group--LI")
        pasos_text = []
        if pasos_section:
            for j, step in enumerate(pasos_section, 1):
                texto = step.get_text(strip=True)
                if texto:
                    pasos_text.append(f"{j}. {texto}")
        df.at[i, 'Preparación'] = ' '.join(pasos_text) if pasos_text else 'No encontrado'

        ### Información Nutricional
        nutrition_table = soup.find("table", class_="mm-recipes-nutrition-facts-summary__table")
        nutricion = []
        if nutrition_table:
            rows = nutrition_table.find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                if len(cols) == 2:
                    label = cols[0].get_text(strip=True)
                    value = cols[1].get_text(strip=True)
                    nutricion.append(f"{label}: {value}")
        df.at[i, 'Información_nutricional'] = ', '.join(nutricion) if nutricion else 'No encontrado'

        ### Detalles adicionales
        detalles = soup.select(".mm-recipes-details__item")
        detalles_text = []
        for item in detalles:
            label = item.select_one(".mm-recipes-details__label")
            value = item.select_one(".mm-recipes-details__value")
            if label and value:
                detalles_text.append(f"{label.get_text(strip=True)}: {value.get_text(strip=True)}")
        df.at[i, 'Detalles'] = ', '.join(detalles_text) if detalles_text else 'No encontrado'

        ### Rating
        rating = soup.select_one("#mm-recipes-review-bar__rating_1-0")
        df.at[i, 'Rating'] = rating.get_text(strip=True) if rating else 'No encontrado'

        ### Comentarios
        comentarios = []
        for r in soup.select(".photo-dialog__item"):
            author = r.select_one(".photo-dialog__profile--linked")
            date = r.select_one(".ugc-review__date")
            text = r.select_one(".ugc-review__text")
            stars = len(r.select(".ugc-review__rating svg.icon-star"))
            if author and date and text and stars > 0:
                comentarios.append(f"{author.text.strip()} ({date.text.strip()} - {stars}/5): {text.text.strip()}")
            if len(comentarios) == 5:
                break
        df.at[i, 'Comentarios'] = '\n'.join(comentarios) if comentarios else 'No se encontraron comentarios'
        ### Imagen
        image_meta = soup.find("meta", property="og:image")
        image_url = image_meta["content"] if image_meta else None
        df.at[i, 'image_url'] = image_url if image_url else 'No disponible'

        if image_url:
            print(f"\n➡️ Imagen de la receta: {df.at[i, 'Nombre']}")
            display(Image(url=image_url, width=300))  # tamaño reducido
        else:
            print("No se encontró imagen.")

    except Exception as e:
        print(f"❌ Error procesando la URL {url}: {e}")
        df.at[i, 'Nombre'] = 'Error'
        df.at[i, 'Ingredientes'] = 'Error'
        df.at[i, 'Image'] = 'Error'
        df.at[i, 'Preparación'] = 'Error'
        df.at[i, 'Descripción'] = 'Error'
        df.at[i, 'Información_nutricional'] = 'Error'
        df.at[i, 'Detalles'] = 'Error'
        df.at[i, 'Rating'] = 'Error'
        df.at[i, 'Comentarios'] = 'Error'

Procesando URLs:   0%|          | 0/100 [00:00<?, ?it/s]


➡️ Imagen de la receta: Air Fryer Jalapeno Popper Pinwheels


/tmp/ipython-input-1859215295.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[i, 'image_url'] = image_url if image_url else 'No disponible'


Procesando URLs:   1%|          | 1/100 [00:00<00:46,  2.13it/s]


➡️ Imagen de la receta: Air Fryer Miso-Glazed Salmon


...

Procesando URLs:  98%|█████████▊| 98/100 [00:53<00:01,  1.66it/s]


➡️ Imagen de la receta: Artichoke Dip Wonton Cups


Procesando URLs:  99%|█████████▉| 99/100 [00:54<00:00,  1.71it/s]


➡️ Imagen de la receta: Garlic Butter Chicken Dump and Bake Casserole


Procesando URLs: 100%|██████████| 100/100 [00:55<00:00,  1.81it/s]


In [18]:
print(df[['Nombre', 'image_url']].head(5))

                                Nombre  \
0  Air Fryer Jalapeno Popper Pinwheels   
1         Air Fryer Miso-Glazed Salmon   
2              Air Fryer Hoisin Salmon   
3            Air Fryer Stuffed Peppers   
4  Air Fryer Croissant Breakfast Boats   

                                           image_url  
0  https://www.allrecipes.com/thmb/hf1VerXbERWTgs...  
1  https://www.allrecipes.com/thmb/w-o2vJurtYbpnR...  
2  https://www.allrecipes.com/thmb/xUBa1jxDj153Jl...  
3  https://www.allrecipes.com/thmb/N6Rm1aSOVSOqzz...  
4  https://www.allrecipes.com/thmb/24ogcOGX6D_ltp...  


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [19]:
def clean_text(raw_text):
    """
    Preprocesa texto eliminando caracteres especiales, palabras vacías (stopwords),
    y dejando solo palabras relevantes.

    Args:
        raw_text (str): Texto original a procesar.

    Returns:
        str: Texto limpio y preprocesado.
    """
    # Convertir a minúsculas
    raw_text = raw_text.lower()
    # Sustituir caracteres especiales por espacios
    raw_text = re.sub(r'[^a-zA-Z]', ' ', raw_text)
    # Tokenizar texto
    tokens = word_tokenize(raw_text)
    # Filtrar palabras que no sean relevantes (stopwords y palabras cortas)
    meaningful_words = filter(lambda word: word not in stop_words and len(word) > 2, tokens)
    return ' '.join(meaningful_words)


In [20]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [21]:
# Combinar columnas relevantes en un solo campo corpus
df['corpus'] = (
    df['Nombre'].fillna('') + ' ' +
    df['Rating'].fillna('') + ' ' +
    df['Descripción'].fillna('') + ' ' +
    df['Ingredientes'].fillna('') + ' ' +
    df['image_url'].fillna('') + ' ' +
    df['Preparación'].fillna('') + ' ' +
    df['Información_nutricional'].fillna('') + ' ' +
    df['Detalles'].fillna('') + ' ' +
    df['Comentarios'].fillna('')
)

# Aplicar función de limpieza a todo el corpus
df['tokens'] = df['corpus'].apply(clean_text)

/tmp/ipython-input-4006254799.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['corpus'] = (
/tmp/ipython-input-4006254799.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tokens'] = df['corpus'].apply(clean_text)


In [22]:
# Generación de embeddings utilizando TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
recipe_embeddings = tfidf_vectorizer.fit_transform(df['tokens'])

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

def search_similar_recipes(user_query, embedding_matrix, recipe_df, tfidf_vectorizer):
    """
    Busca recetas similares a una consulta proporcionada por el usuario usando similitud de coseno.

    """

    # Preprocess the user query
    processed_query = clean_text(user_query)
    # tfidf expects a string, not a list of tokens
    joined_query = " ".join(processed_query.split())

    # Vectorize the query
    query_embedding = tfidf_vectorizer.transform([joined_query])

    # Calculate cosine similarities
    similarity_scores = cosine_similarity(query_embedding, embedding_matrix).flatten()

    # Add similarity scores to a copy of the DataFrame
    recipe_df = recipe_df.copy()
    recipe_df['similarity'] = similarity_scores

    # Sort and select the top 10 recipes
    top_recipes = recipe_df.sort_values(by='similarity', ascending=False).head(10)

    # Return all relevant columns for printing
    return top_recipes[['Nombre', 'Ingredientes', 'image_url', 'Preparación', 'Información_nutricional', 'Rating', 'similarity', 'Descripción', 'Detalles', 'Comentarios', 'Receta_URL', 'cluster']]

In [24]:
def cluster_recipes(embedding_matrix, num_clusters=10):
    """
    Realiza agrupamiento (clustering) de recetas basado en los embeddings generados.

    Args:
        embedding_matrix (sparse matrix): Matriz de embeddings generada.
        num_clusters (int): Número de clusters a formar.

    Returns:
        list: Etiquetas de cluster asignadas a cada receta.
    """
    print("Realizando clustering con K-means...")
    kmeans_model = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans_model.fit_predict(embedding_matrix)
    return cluster_labels

# Agregar etiquetas de cluster a los datos
df['cluster'] = cluster_recipes(recipe_embeddings, num_clusters=10)

Realizando clustering con K-means...


/tmp/ipython-input-3356880454.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cluster'] = cluster_recipes(recipe_embeddings, num_clusters=10)


In [27]:
def example_query_run(user_query, recipe_df, tfidf_vectorizer):
    """
    Ejecuta una consulta de ejemplo y muestra los resultados completos.

    Args:
        user_query (str): Consulta del usuario.
        recipe_df (DataFrame): DataFrame con las recetas ya procesadas.
        tfidf_vectorizer (TfidfVectorizer): Vectorizador entrenado.
    """
    results = search_similar_recipes(user_query, recipe_embeddings, recipe_df, tfidf_vectorizer)
    print("Recetas similares encontradas:\n")

    for _, row in results.iterrows():
        ingredientes = '\n'.join(f"- {i.strip()}" for i in row['Ingredientes'].split(','))
        preparacion = '\n'.join(f"{idx+1}. {p.strip()}" for idx, p in enumerate(row['Preparación'].split('.')) if p.strip())

        print(f"""
Receta: {row['Nombre']}
URL: {row.get('Receta_URL', 'No disponible')}
Similitud: {row['similarity']:.4f}
Cluster: {row.get('cluster', 'N/A')}

Rating: {row['Rating']}

Descripción:
{row['Descripción']}

Ingredientes:
{ingredientes}

Pasos:
{preparacion}
""")
        if row.get('image_url') and row['image_url'] != 'No disponible':
            print("Imagen:")
            display(Image(url=row['image_url'], width=300))
        else:
            print("Imagen no disponible")

        print(f"""

Información nutricional:
{row['Información_nutricional']}

Detalles:
{row['Detalles']}

Comentarios:
{row['Comentarios']}

{'='*60}
""")

# Ejecutar consulta
user_query_example = "chicken with lemon"
example_query_run(user_query_example, df, tfidf_vectorizer)

Recetas similares encontradas:


Receta: Lemon Garlic Butter Chicken Spiedini
URL: https://www.allrecipes.com/lemon-garlic-butter-chicken-spiedini-recipe-8727930
Similitud: 0.4758
Cluster: 0

Rating: 4.8

Descripción:
These lemon garlic butter chicken spiedini are air-fried crispy, golden breaded chicken skewers with a light lemon garlic butter drizzle.

Ingredientes:
- 1/2cupextra-virgin olive oil
- 1/4cupwhite wine
- such as Pinot Grigio
- 3tablespoonslemon juice
- divided
- 1teaspoonlemon zest
- 4clovesgarlic
- finely minced
- divided
- 1/4teaspooncrushed red pepper flakes
- 1/4teaspoondriedoregano
- 2teaspoonskosher salt
- divided
- 1teaspoonfreshly ground blackpepper
- divided
- 2poundsskinless boneless chicken thighs
- cut into 1 1/2 inch pieces
- 2/3cupItalian breadcrumbs
- 2/3cuppanko
- 1/2cupgrated Parmesan cheese
- 1/2teaspoongarlic powder
- 6 (6 to 8 inch)skewers
- olive oilcooking spray
- 4tablespoonsbutter
- 2teaspoonsfinely choppedfresh parsley

Pasos:
1. 1
2. Whisk toget



Información nutricional:
637: Calories, 41g: Fat, 22g: Carbs, 43g: Protein

Detalles:
Prep Time:: 15 mins, Cook Time:: 15 mins, Total Time:: 30 mins, Servings:: 6

Comentarios:
No se encontraron comentarios



Receta: Greek Lemon Chicken Orzo Casserole
URL: https://www.allrecipes.com/greek-lemon-chicken-orzo-casserole-recipe-11880811
Similitud: 0.4636
Cluster: 0

Rating: No encontrado

Descripción:
This Greek lemon chicken orzo casserole features Greek-seasoned chicken, tomatoes, zucchini, and olives in a cozy one-dish dinner.

Ingredientes:
- 1/2cupuncooked orzo
- 2teaspoonsolive oil
- 1 1/2teaspoonsDijon mustard
- 2teaspoonslow-salt Greek seasoning blend
- 3/4poundskinless
- bonelss chicken thighs
- cut into 1-inch pieces
- 1cupcherry tomatoes
- halved
- 1/2cup choppedyellow bell pepper
- or any color bell pepper
- 1smallzucchini
- chopped
- 2tablespoonsslicedblack olives
- 1/3cupcrumbledfeta cheese
- divided
- 1cuplow-sodium chicken broth
- 2tablespoonsGreek yogurt
- 2tablespoonsc



Información nutricional:
455: Calories, 18g: Fat, 38g: Carbs, 39g: Protein

Detalles:
Prep Time:: 15 mins, Cook Time:: 35 mins, Stand Time:: 10 mins, Total Time:: 1 hr, Servings:: 3

Comentarios:
No se encontraron comentarios



Receta: Air Fryer Lemon Garlic Parmesan Chicken
URL: https://www.allrecipes.com/air-fryer-lemon-garlic-parmesan-chicken-recipe-8726749
Similitud: 0.4098
Cluster: 0

Rating: 5.0

Descripción:
These simple lemon garlic Parmesan chicken thighs are cooked in the air fryer. Seasoned with lemon zest, garlic, and oregano and coated in Parmesan cheese they come out juicy and tender

Ingredientes:
- 1 1/2poundsskinless boneless chicken thighs
- 3clovesgarlic
- minced
- 1teaspoonlemon zest
- 1teaspoonpaprika
- 1teaspoondried oregano
- 1/2teaspoonsalt
- 1/4teaspooncrushed red pepper
- 1/2cupfreshlygrated parmesan cheese
- 1/4cuppanko bread crumbs
- cooking spray

Pasos:
1. 1
2. Gather all ingredients
3. Dotdash Meredith Food Studios 2
4. Preheat an air fryer to 400 degr



Información nutricional:
365: Calories, 17g: Fat, 8g: Carbs, 46g: Protein

Detalles:
Prep Time:: 20 mins, Cook Time:: 12 mins, Total Time:: 32 mins, Servings:: 4

Comentarios:
No se encontraron comentarios



Receta: Air Fryer Chicken Bites
URL: https://www.allrecipes.com/air-fryer-chicken-bites-recipe-8599352
Similitud: 0.3728
Cluster: 0

Rating: 4.0

Descripción:
These air fryer chicken bites, seasoned with lemon, garlic, and Italian herbs, are a perfect addition to an appetizer tray, great to add to a salad, or as a main dish for a family dinner.

Ingredientes:
- 1poundskinless
- boneless chicken breasts
- cut into bite-sized pieces
- 3tablespoonsbutter
- melted
- 1tablespoonolive oil
- 1teaspoonlemon juice
- or to taste
- 3clovesgarlic
- minced
- 1teaspoonItalian seasoning
- 1teaspoonsalt
- 1teaspoonpaprika
- 1/2teaspooncayenne pepper (optional)

Pasos:
1. 1
2. Preheat an air fryer to 350 degrees F (175 degrees C) setting  for 5 minutes
3. 2
4. Stir melted butter, olive oil, lemo



Información nutricional:
299: Calories, 16g: Fat, 1g: Carbs, 36g: Protein

Detalles:
Prep Time:: 10 mins, Cook Time:: 10 mins, Total Time:: 20 mins, Servings:: 4

Comentarios:
No se encontraron comentarios



Receta: The 2-Minute Mediterranean Salad Dressing We Can't Stop Making
URL: https://www.allrecipes.com/lemon-tahini-dressing-recipe-11701927
Similitud: 0.3244
Cluster: 5

Rating: No encontrado

Descripción:
This lemon tahini dressing made with olive oil, lemon juice, tahini paste, and garlic will add a bright and deliciously tangy flavor to any salad!

Ingredientes:
- ½lemon
- juiced
- 3tablespoonstahini paste
- 2tablespoonsolive oil
- 2tablespoonswater
- 1clovegarlic
- roughly chopped
- ½teaspoonsalt
- ¼teaspoonfreshly ground black pepper

Pasos:
1. 1
2. Gather the ingredients
3. Allrecipes/France Cevallos 2
4. Combine lemon juice, tahini, olive oil, water, garlic, salt, and pepper in a mini food processor or blender; process until smooth
5. Allrecipes/France Cevallos

Imagen:


...



Información nutricional:
240: Calories, 4g: Fat, 7g: Carbs, 41g: Protein

Detalles:
Prep Time:: 10 mins, Cook Time:: 20 mins, Marinate Time:: 8 hrs, Total Time:: 8 hrs 30 mins, Servings:: 4

Comentarios:
No se encontraron comentarios



Receta: Air Fryer Honey-Mustard Chicken Thighs
URL: https://www.allrecipes.com/air-fryer-honey-mustard-chicken-thighs-recipe-7970816
Similitud: 0.2457
Cluster: 0

Rating: 4.6

Descripción:
These air fryer honey-mustard chicken thighs prove that simple chicken recipes don’t have to be boring! Jazz up your chicken thighs with this tasty, smoky, honey-mustard sauce.

Ingredientes:
- 4boneless skinless chicken thighs
- 1/2teaspoononion powder
- 1/2teaspoongarlic powder
- 1/2teaspoonsmoked paprika
- 1/4teaspoonsalt
- or to taste
- 1/4teaspoonfreshly groundblack pepper
- or to taste
- 2tablespoonshoney
- 2tablespoonsDijon mustard
- olive oil cooking spray

Pasos:
1. 1
2. Preheat the air fryer to 390 degrees F (198 degrees C), if recommended by the manufactu



Información nutricional:
364: Calories, 20g: Fat, 10g: Carbs, 38g: Protein

Detalles:
Prep Time:: 5 mins, Cook Time:: 15 mins, Total Time:: 20 mins, Servings:: 4

Comentarios:
No se encontraron comentarios



Receta: Easy Air Fryer Chicken Breast
URL: https://www.allrecipes.com/recipe/283850/easy-air-fried-chicken-breast/
Similitud: 0.2387
Cluster: 0

Rating: 4.9

Descripción:
This air fryer chicken breast is perfectly seasoned outside and air fried until golden, tender, and incredibly juicy inside for a quick and easy dinner.

Ingredientes:
- 1(8 ounce)chicken breast
- 2teaspoonsolive oil
- 1/4teaspoongarlic powder
- or to taste
- salt and freshly ground black pepper to taste

Pasos:
1. 1
2. Gather all ingredients
3. Preheat an air fryer to 360 degrees F (182 degrees C)
4. Allrecipes / Qi Ai 2
5. Brush both sides of chicken breast with olive oil; season with garlic powder, salt, and pepper on one side
6. Place chicken breast, seasoned-side down, into the air fryer basket and sprinkl



Información nutricional:
456: Calories, 17g: Fat, 1g: Carbs, 70g: Protein

Detalles:
Prep Time:: 5 mins, Cook Time:: 20 mins, Additional Time:: 5 mins, Total Time:: 30 mins, Servings:: 1, Yield:: 1 chicken breast

Comentarios:
No se encontraron comentarios



Receta: Chicken au Poivre
URL: https://www.allrecipes.com/chicken-au-poivre-recipe-11861726
Similitud: 0.2358
Cluster: 0

Rating: 5.0

Descripción:
In this chicken au poivre, cracked peppercorns add a punch of spice to a lovely cream sauce for chicken thighs. It's especially nice served over rice or noodles.

Ingredientes:
- 1tablespoonblackpeppercorns
- 1tablespoonolive oil
- 3tablespoonsunsalted butter
- divided
- 2poundsskinless
- boneless chicken thighs
- patted dry
- 1/2teaspoonsalt
- or to taste
- 2tablespoonsmincedshallot
- 1cuplow-sodium chicken broth
- 1/2cupheavy cream
- 3fresh thyme sprigs
- or to taste
- 1tablespoonfreshly squeezedlemon juice
- fresh parsley sprigsfor garnish (optional)

Pasos:
1. 1
2. Add peppercorn



Información nutricional:
598: Calories, 41g: Fat, 4g: Carbs, 57g: Protein

Detalles:
Prep Time:: 10 mins, Cook Time:: 30 mins, Total Time:: 40 mins, Servings:: 4

Comentarios:
No se encontraron comentarios



Receta: Air Fryer Chicken Parmesan
URL: https://www.allrecipes.com/air-fryer-chicken-parmesan-recipe-8698442
Similitud: 0.2221
Cluster: 0

Rating: 4.6

Descripción:
The air fryer is perfect for making chicken Parmesan. No need to fuss with deep-frying and the chicken turns out cruspy and golden brown with lots of cheese and sauce.

Ingredientes:
- 2(8-ounce)boneless skinless chicken breasts
- patted dry
- 3/4teaspoonkosher salt
- divided
- 1/2teaspoonfreshly groundblack pepper
- divided
- 1/3cupall purpose flour
- 2largeeggs
- lightly beaten
- 1 1/2cupsseasoned panko (Japanese-style breadcrumbs)
- 1/4cupfreshly grated Parmesan cheese
- plus more for garnish
- 2tablespoonsolive oil
- 1/4teaspooncrushed red pepper
- 1/4teaspoongarlic powder
- 1cupjarredmarinara sauce
- plus more f



Información nutricional:
582: Calories, 18g: Fat, 45g: Carbs, 57g: Protein

Detalles:
Prep Time:: 15 mins, Cook Time:: 25 mins, Total Time:: 40 mins, Servings:: 4

Comentarios:
No se encontraron comentarios


